# Load Libraries & Data

In [0]:
# Core Libraries
import pandas as pd
import numpy as np
import re

In [0]:
df = pd.read_csv("/Volumes/workspace/default/ai-enhanced_project/Cars Datasets 2025.csv", encoding='latin-1')
print(f"Starting shape: {df.shape}")
df

Starting shape: (1218, 11)


,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm
...,...,...,...,...,...,...,...,...,...,...,...
1213,Toyota,Crown Signia,2.5L Hybrid I4,2487 cc,240 hp,180 km/h,7.6 sec,"$43,590  $48,000",Hybrid (Gas + Electric),5,239 Nm
1214,Toyota,4Runner (6th Gen),2.4L Turbo I4 (i-FORCE MAX Hybrid),2393 cc + Battery,326 hp,180 km/h,6.8 sec,"$50,000",Hybrid,7,630 Nm
1215,Toyota,Corolla Cross,2.0L Gas / 2.0L Hybrid,1987 cc / Hybrid batt,169  196 hp,190 km/h,8.0  9.2 sec,"$25,210  $29,135",Gas / Hybrid,5,190  210 Nm
1216,Toyota,C-HR+,1.8L / 2.0L Hybrid,1798 / 1987 cc + batt,140  198 hp,180 km/h,7.9  10.5 sec," 33,000",Hybrid,5,190  205 Nm


# Clean Data

Check for NullValues

In [0]:
df.isnull().sum()

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          3
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    6
Cars Prices                  0
Fuel Types                   0
Seats                        0
Torque                       1
dtype: int64

Check for Duplicates

In [0]:
print(df.duplicated().sum())

4


General Cleaning

In [0]:
# Remove duplicate rows
df = df.drop_duplicates()

# Drop columns with more than 50% missing values
threshold = len(df) * 0.5
df = df.dropna(axis=1, thresh=threshold)

# Fill remaining missing values with column median (numeric) or mode (categorical)
for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Remove rows with impossible values (e.g., negative prices or mileage)
if 'Price' in df.columns:
    df = df[df['Price'] >= 0]
if 'Mileage' in df.columns:
    df = df[df['Mileage'] >= 0]

print(f"Cleaned shape: {df.shape}")
df.head()

Cleaned shape: (1214, 11)


,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm


In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1214 entries, 0 to 1217
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Company Names              1214 non-null   object
 1   Cars Names                 1214 non-null   object
 2   Engines                    1214 non-null   object
 3   CC/Battery Capacity        1214 non-null   object
 4   HorsePower                 1214 non-null   object
 5   Total Speed                1214 non-null   object
 6   Performance(0 - 100 )KM/H  1214 non-null   object
 7   Cars Prices                1214 non-null   object
 8   Fuel Types                 1214 non-null   object
 9   Seats                      1214 non-null   object
 10  Torque                     1214 non-null   object
dtypes: object(11)
memory usage: 113.8+ KB


In [0]:
display(df)

Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm
BMW,Mclaren 720s,V8,"3,994 cc",710 hp,341 km/h,2.9 sec,"$499,000",Petrol,2,770 Nm
ASTON MARTIN,VANTAGE F1,V8,"3,982 cc",656 hp,314 km/h,3.6 sec,"$193,440",Petrol,2,685 Nm
BENTLEY,Continental GT Azure,V8,"3,996 cc",550 hp,318 km/h,4.0 sec,"$311,000",Petrol,4,900 Nm
LAMBORGHINI,VENENO ROADSTER,V12,"6,498 cc",750 hp,356 km/h,2.9 sec,"$4,500,000",Petrol,2,690 Nm
FERRARI,F8 TRIBUTO,V8,"3,900 cc",710 hp,340 km/h,2.9 sec,"$280,000",Petrol,2,770 Nm


In [0]:
df

,Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
0,FERRARI,SF90 STRADALE,V8,3990 cc,963 hp,340 km/h,2.5 sec,"$1,100,000",plug in hyrbrid,2,800 Nm
1,ROLLS ROYCE,PHANTOM,V12,6749 cc,563 hp,250 km/h,5.3 sec,"$460,000",Petrol,5,900 Nm
2,Ford,KA+,1.2L Petrol,"1,200 cc",70-85 hp,165 km/h,10.5 sec,"$12,000-$15,000",Petrol,5,100 - 140 Nm
3,MERCEDES,GT 63 S,V8,"3,982 cc",630 hp,250 km/h,3.2 sec,"$161,000",Petrol,4,900 Nm
4,AUDI,AUDI R8 Gt,V10,"5,204 cc",602 hp,320 km/h,3.6 sec,"$253,290",Petrol,2,560 Nm
...,...,...,...,...,...,...,...,...,...,...,...
1213,Toyota,Crown Signia,2.5L Hybrid I4,2487 cc,240 hp,180 km/h,7.6 sec,"$43,590  $48,000",Hybrid (Gas + Electric),5,239 Nm
1214,Toyota,4Runner (6th Gen),2.4L Turbo I4 (i-FORCE MAX Hybrid),2393 cc + Battery,326 hp,180 km/h,6.8 sec,"$50,000",Hybrid,7,630 Nm
1215,Toyota,Corolla Cross,2.0L Gas / 2.0L Hybrid,1987 cc / Hybrid batt,169  196 hp,190 km/h,8.0  9.2 sec,"$25,210  $29,135",Gas / Hybrid,5,190  210 Nm
1216,Toyota,C-HR+,1.8L / 2.0L Hybrid,1798 / 1987 cc + batt,140  198 hp,180 km/h,7.9  10.5 sec," 33,000",Hybrid,5,190  205 Nm


Clean individual cells
E.g. Making "3,900 cc" into "3900"

In [0]:
def clean_capacity(cell):
    if not isinstance(cell, str):
        return np.nan
    numbers = re.findall(r'\d+(?:\.\d+)?', cell.replace(",", ""))
    numbers = [float(num) for num in numbers]
    return sum(numbers) / len(numbers) if numbers else np.nan

In [0]:
df['CC/Battery Capacity'] = df['CC/Battery Capacity'].apply(clean_capacity)

In [0]:
def clean_numeric_cell(cell):
    if pd.isnull(cell):
        return np.nan
    cell = str(cell).replace(",", "")
    numbers = re.findall(r'\d+(?:\.\d+)?', cell)
    numbers = [float(num) for num in numbers]
    return sum(numbers) / len(numbers) if numbers else np.nan

# List of columns to clean
columns_to_clean = [
    'HorsePower',
    'Total Speed',
    'Performance(0 - 100 )KM/H',
    'Cars Prices',
    'Seats',
    'Torque'
]

In [0]:
# Apply the function to each column
for col in columns_to_clean:
    df[col] = df[col].apply(clean_numeric_cell)

In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1214 entries, 0 to 1217
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Company Names              1214 non-null   object 
 1   Cars Names                 1214 non-null   object 
 2   Engines                    1214 non-null   object 
 3   CC/Battery Capacity        1212 non-null   float64
 4   HorsePower                 1214 non-null   float64
 5   Total Speed                1214 non-null   float64
 6   Performance(0 - 100 )KM/H  1214 non-null   float64
 7   Cars Prices                1213 non-null   float64
 8   Fuel Types                 1214 non-null   object 
 9   Seats                      1214 non-null   float64
 10  Torque                     1214 non-null   float64
dtypes: float64(7), object(4)
memory usage: 113.8+ KB


In [0]:
df.isna().sum()

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          2
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    0
Cars Prices                  1
Fuel Types                   0
Seats                        0
Torque                       0
dtype: int64

In [0]:
df.dropna(inplace=True)

Drop Empty Cells

In [0]:
df.duplicated().sum()

np.int64(1)

Drop duplicates

In [0]:
df.drop_duplicates(inplace=True)

Correct types

In [0]:
df['CC/Battery Capacity'] = df['CC/Battery Capacity'].astype(int)
df['HorsePower'] = df['HorsePower'].astype(int)
df['Total Speed'] = df['Total Speed'].astype(int)
df['Performance(0 - 100 )KM/H'] = df['Performance(0 - 100 )KM/H'].astype(float)
df['Cars Prices'] = df['Cars Prices'].astype(int)
df['Seats'] = df['Seats'].astype(int)
df['Torque'] = df['Torque'].astype(int)

In [0]:
print(df['Fuel Types'].unique())

['plug in hyrbrid' 'Petrol' 'Diesel' 'Hybrid' 'Electric' 'Petrol/Diesel'
 'Plug-in Hybrid' 'Petrol/AWD' 'Petrol/Hybrid' 'Hydrogen' 'Diesel/Petrol'
 'Petrol/EV' 'Hybrid/Electric' 'Petrol, Hybrid' 'Petrol, Diesel'
 'Hybrid (Petrol)' 'CNG/Petrol' 'Hybrid/Petrol' 'Diesel Hybrid'
 'Hybrid (Gas + Electric)' 'Gas / Hybrid' 'Hybrid / Plug-in']


In [0]:
fuel_category_map = {
        'plug in hyrbrid': 'Hybrid',         # likely typo
        'Petrol': 'Petrol',
        'Diesel': 'Diesel',
        'Hybrid': 'Hybrid',
        'Electric': 'Electric',
        'Petrol/Diesel': 'Petrol',
        'Plug-in Hybrid': 'Hybrid',
        'Petrol/AWD': 'Petrol',
        'Petrol/Hybrid': 'Hybrid',
        'Hydrogen': 'Hydrogen',                 # can exclude or treat separately
        'Diesel/Petrol': 'Diesel',
        'Petrol/EV': 'Petrol',
        'Hybrid/Electric': 'Hybrid',
        'Petrol, Hybrid': 'Hybrid',
        'Petrol, Diesel': 'Petrol',
        'Hybrid (Petrol)': 'Hybrid',
        'CNG/Petrol': 'Hybrid',
        'Hybrid/Petrol': 'Hybrid',
        'Diesel Hybrid': 'Hybrid',
        'Hybrid (Gas + Electric)': 'Hybrid',
        'Gas / Hybrid': 'Hybrid',
        'Hybrid / Plug-in': 'Hybrid'
}

In [0]:
df['Fuel Types'] = df['Fuel Types'].map(fuel_category_map)

In [0]:
display(df)

Company Names,Cars Names,Engines,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Fuel Types,Seats,Torque
FERRARI,SF90 STRADALE,V8,3990,963,340,2.5,1100000,Hybrid,2,800
ROLLS ROYCE,PHANTOM,V12,6749,563,250,5.3,460000,Petrol,5,900
Ford,KA+,1.2L Petrol,1200,77,165,10.5,13500,Petrol,5,120
MERCEDES,GT 63 S,V8,3982,630,250,3.2,161000,Petrol,4,900
AUDI,AUDI R8 Gt,V10,5204,602,320,3.6,253290,Petrol,2,560
BMW,Mclaren 720s,V8,3994,710,341,2.9,499000,Petrol,2,770
ASTON MARTIN,VANTAGE F1,V8,3982,656,314,3.6,193440,Petrol,2,685
BENTLEY,Continental GT Azure,V8,3996,550,318,4.0,311000,Petrol,4,900
LAMBORGHINI,VENENO ROADSTER,V12,6498,750,356,2.9,4500000,Petrol,2,690
FERRARI,F8 TRIBUTO,V8,3900,710,340,2.9,280000,Petrol,2,770


In [0]:
df['Company Names'] = df['Company Names'].str.strip().str.upper()

Remove Outliers

In [0]:
def outliers(d,value):
    Q1 = d[value].quantile(0.25)
    Q3 = d[value].quantile(0.75)
    IQR = Q3 - Q1
    
    outliers = d[(d[value] > (Q1 - (1.5 * IQR))) & (d[value] < (Q3 + 1.5 * IQR))]
    return outliers

In [0]:
df = outliers(df,'Cars Prices')
df = outliers(df,'HorsePower')

Save data under new name to indicate it's been cleaned

In [0]:
# Rename columns to remove invalid characters for Delta table
df_renamed = df.rename(columns={
    'Company Names': 'company_names',
    'Cars Names': 'cars_names',
    'CC/Battery Capacity': 'cc_battery_capacity',
    'Total Speed': 'total_speed',
    'Performance(0 - 100 )KM/H': 'performance_0_100_kmh',
    'Cars Prices': 'cars_prices',
    'Fuel Types': 'fuel_types'
})

df_spark = spark.createDataFrame(df_renamed)
df_spark.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/ai-enhanced_project/Cars_Datasets_2025_cleaned_delta")

In [0]:
df_renamed


,company_names,cars_names,Engines,cc_battery_capacity,HorsePower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,Seats,Torque
2,FORD,KA+,1.2L Petrol,1200,77,165,10.50,13500,Petrol,5,120
17,TOYOTA,GR SUPRA,I4,2998,382,250,4.10,53900,Petrol,2,500
18,TOYOTA,TOYOTA 86,BOXER-4,1998,205,226,6.40,27000,Petrol,2,205
19,TOYOTA,TOYOTA GR86,BOXER-4,2387,228,226,5.60,30000,Petrol,4,250
20,TOYOTA,TOYOTA LAND CRUISER,V8,5663,381,220,6.70,85000,Diesel,7,500
...,...,...,...,...,...,...,...,...,...,...,...
1213,TOYOTA,Crown Signia,2.5L Hybrid I4,2487,240,180,7.60,45795,Hybrid,5,239
1214,TOYOTA,4Runner (6th Gen),2.4L Turbo I4 (i-FORCE MAX Hybrid),2393,326,180,6.80,50000,Hybrid,7,630
1215,TOYOTA,Corolla Cross,2.0L Gas / 2.0L Hybrid,1987,182,190,8.60,27172,Hybrid,5,200
1216,TOYOTA,C-HR+,1.8L / 2.0L Hybrid,1892,169,180,9.20,33000,Hybrid,5,197


In [0]:
df.to_csv("/Volumes/workspace/default/ai-enhanced_project/Cars Datasets 2025 cleaned.csv", index=False)